# Triage_HF Statistical & Graphical Analytical Model
* Started date: 02/07/2024 - 05:09 AM
* Data Management team, Triage_HF

# 1. Dataset cleaning
We will import our dataset and perform a first cleanup to begin to understand which columns and values we are dealing with.

In [14]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option('display.max.rows', None)
pd.set_option('display.max.columns', None)

In [15]:
train_df = pd.read_csv('../dataset/raw/TRIAGE 2024.csv')

In [16]:
train_df.head()

,3+-99999|a,[ñ_MJ,NOMBRE Y APELLIDO,MOTIVO DE CONSULTA,BOX,TRIAGE,ENFERMERO,MEDICO,DESTINO,A,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,FECHA: 01/01/2024 ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,01-01,LUSI,FIEBRE Y TOS,18,IV,ERIKA,RODRIGO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,01-01,LO PINTO CARLOS,FIEBRE Y TOS,17,IV,SOLEDAD G,SOLEDAD,alta,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,01-01,BANDERA,TOS,12,IV,ERIKA,RODRIGO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,01-01,TOMINO EDUARDO,HTA,12,IV,SOLEDAD G,SOLEDAD,ALTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


At first, you can see how certain columns are wrongly named, so I will assign a corresponding name to them:

In [17]:
cols_rename = { '[ñ_MJ': 'FECHA DE INGRESO',
                'A': 'AISLADO' }
train_df.rename(columns = cols_rename, inplace = True)

In addition, each time you move to the next day according to the date of entry, the first column returns to 1:

In [18]:
train_df.loc[train_df['FECHA DE INGRESO'] == '01-01', '3+-99999|a'].iloc[0] == train_df.loc[train_df['FECHA DE INGRESO'] == '01-02', '3+-99999|a'].iloc[0]

True

Therefore, I will name that first column "NUMERO DE TURNO" in reference to the fact that the turns are reset at the beginning of the next day:

In [19]:
col_rename = { '3+-99999|a': 'NUMERO DE TURNO' }
train_df.rename(columns = col_rename, inplace = True)

We will remove the header that appears every time a new day begins:

In [20]:
train_df = train_df[train_df['NUMERO DE TURNO'].str.contains('FECHA') == False]

For possible machine learning models in the future, the triage level should be int dtype:

In [21]:
vals_rename = { 'I': 1, 'II': 2, 'III': 3, 'IV': 4 }
train_df['TRIAGE'] = train_df['TRIAGE'].replace(vals_rename)

In [22]:
train_df['TRIAGE'].dtype

dtype('O')

However, for the moment we will leave it in object type because it has missing values, and then we will decide what to do with them:

In [23]:
train_df['TRIAGE'].isnull().count()

1634

Finally, empty and unnamed columns can be visualized. We will remove them:

In [24]:
cols_to_keep = ['NUMERO DE TURNO', 'FECHA DE INGRESO', 'NOMBRE Y APELLIDO', 'MOTIVO DE CONSULTA', 'BOX', 'TRIAGE', 'ENFERMERO', 'MEDICO', 'DESTINO', 'AISLADO']
train_df = train_df[cols_to_keep]

This is how our dataframe would look at first:

In [27]:
train_df.head()

,NUMERO DE TURNO,FECHA DE INGRESO,NOMBRE Y APELLIDO,MOTIVO DE CONSULTA,BOX,TRIAGE,ENFERMERO,MEDICO,DESTINO,AISLADO
1,1,01-01,LUSI,FIEBRE Y TOS,18,4,ERIKA,RODRIGO,NaN,NaN
2,2,01-01,LO PINTO CARLOS,FIEBRE Y TOS,17,4,SOLEDAD G,SOLEDAD,alta,NaN
3,3,01-01,BANDERA,TOS,12,4,ERIKA,RODRIGO,NaN,NaN
4,4,01-01,TOMINO EDUARDO,HTA,12,4,SOLEDAD G,SOLEDAD,ALTA,NaN
5,5,01-01,D IORIO ROLANDO EMILIO,FIEBRE,5,4,ERIKA,SOLEDAD,NaN,NaN


In [39]:
def turnos_fecha(desde, hasta):
    train_df_acotado = train_df[train_df['FECHA DE INGRESO'] != np.nan]
    train_df_acotado = pd.to_datetime(train_df_acotado['FECHA DE INGRESO'], format='%d-%m', errors='coerce')
    print(train_df_acotado)
    #train_df_acotado = train_df_acotado[(train_df_acotado['FECHA DE INGRESO'] >= desde) & (train_df_acotado['FECHA DE INGRESO'] <= hasta)]
    #cantidad_pacientes = train_df_acotado.groupby('FECHA DE INGRESO')['NUMERO DE TURNO'].count().reset_index()
    #print(cantidad_pacientes)
    #fig = px.bar(cantidad_pacientes, x='FECHA DE INGRESO', y='NUMERO DE TURNO', barmode="group")
    #fig.show()
    
turnos_fecha("01-01", "02-01")

1      1900-01-01
2      1900-01-01
3      1900-01-01
4      1900-01-01
5      1900-01-01
6      1900-01-01
7      1900-01-01
8      1900-01-01
9      1900-01-01
10     1900-01-01
11     1900-01-01
12     1900-01-01
13     1900-01-01
14     1900-01-01
15     1900-01-01
16     1900-01-01
17     1900-01-01
18     1900-01-01
20     1900-01-02
21     1900-01-02
22     1900-01-02
23     1900-01-02
24     1900-01-02
25     1900-01-02
26     1900-01-02
27     1900-01-02
28     1900-01-02
29     1900-01-02
30     1900-01-02
31     1900-01-02
32     1900-01-02
33     1900-01-02
34     1900-01-02
35     1900-01-02
36     1900-01-02
37     1900-01-02
38     1900-01-02
39     1900-01-02
40     1900-01-02
41     1900-01-02
42     1900-01-02
43     1900-01-02
44     1900-01-02
45     1900-01-02
46     1900-01-02
47     1900-01-02
48     1900-01-02
49     1900-01-02
50     1900-01-02
51     1900-01-02
52     1900-01-02
53     1900-01-02
54     1900-01-02
55     1900-01-02
56     1900-01-02
57     190

In [32]:
print(train_df.loc[24])

NUMERO DE TURNO                       5
FECHA DE INGRESO                  02-01
NOMBRE Y APELLIDO     RIPAMONTI GUSTAVO
MOTIVO DE CONSULTA       DOLOR DE PECHO
BOX                                  17
TRIAGE                                2
ENFERMERO                        SILVIA
MEDICO                          MARTIN 
DESTINO                             NaN
AISLADO                              NO
Name: 24, dtype: object
